# Install & Import Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import matplotlib.pyplot as plt
import os

# Device Configuration

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/GenAI/Lab 7/"


# Image Loader

In [ ]:
image_size = 512 if torch.cuda.is_available() else 256

loader = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def load_image(image_path):
    image = Image.open(image_path).convert("RGB")
    image = loader(image).unsqueeze(0)
    return image.to(device, torch.float)

content_image = load_image(BASE_DIR+"eye.jpg")
style_image = load_image(BASE_DIR+"style2.jpg")

# Display Images

In [ ]:
def imshow(tensor, title=None):
    image = tensor.clone().detach().cpu().squeeze(0)

    # De-normalize
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

    image = image * std + mean
    image = torch.clamp(image, 0, 1)

    image = transforms.ToPILImage()(image)

    plt.imshow(image)
    if title:
        plt.title(title)
    plt.axis("off")

In [ ]:
plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
imshow(content_image, "Content Image")

plt.subplot(1,2,2)
imshow(style_image, "Style Image")

plt.show()

# Load Pretrained VGG19 Model

In [ ]:
vgg = models.vgg19(pretrained=True).features.to(device).eval()

for param in vgg.parameters():
    param.requires_grad = False

#  Feature Extraction

In [ ]:
def get_features(image, model):
    layers = {
        '0': 'conv_1',
        '5': 'conv_2',
        '10': 'conv_3',
        '19': 'conv_4',
        '28': 'conv_5'
    }

    features = {}
    x = image

    for name, layer in model._modules.items():
        x = layer(x)
        if name in layers:
            features[layers[name]] = x

    return features

content_features = get_features(content_image, vgg)
style_features = get_features(style_image, vgg)

# Gram Matrix

In [ ]:
def gram_matrix(tensor):
    _, d, h, w = tensor.size()
    tensor = tensor.view(d, h*w)
    gram = torch.mm(tensor, tensor.t())
    return gram

# Target Image

In [ ]:
target = content_image.clone().requires_grad_(True).to(device)

# Optimizer & Weights

In [ ]:
optimizer = optim.LBFGS([target])

content_layer = 'conv_4'

style_layers = ['conv_1', 'conv_2', 'conv_3', 'conv_4', 'conv_5']

style_weights = {
    'conv_1': 1.0,
    'conv_2': 0.8,
    'conv_3': 0.5,
    'conv_4': 0.3,
    'conv_5': 0.1
}

content_weight = 5e3
style_weight = 2e5

In [ ]:
output_dir = BASE_DIR+"nst_outputs"
os.makedirs(output_dir, exist_ok=True)

print("Folder created:", output_dir)

# Training Loop

In [ ]:
steps = 2000
run = [0]

while run[0] <= steps:

    def closure():
        optimizer.zero_grad()

        target_features = get_features(target, vgg)

        content_loss = torch.mean(
            (target_features[content_layer] - content_features[content_layer])**2
        )

        style_loss = 0
        for layer in style_layers:
            target_feature = target_features[layer]
            target_gram = gram_matrix(target_feature)

            style_feature = style_features[layer]
            style_gram = gram_matrix(style_feature)

            _, d, h, w = target_feature.shape

            layer_style_loss = style_weights[layer] * torch.mean(
                (target_gram - style_gram)**2
            ) / (d*h*w)

            style_loss += layer_style_loss

        total_loss = content_weight * content_loss + style_weight * style_loss

        total_loss.backward()


        if run[0] % 100 == 0:
            print(f"Step {run[0]}, Total Loss: {total_loss.item()}")
            temp = target.clone().detach().cpu().squeeze(0)
            mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
            std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
            temp = temp * std + mean
            temp = torch.clamp(temp, 0, 1)

            temp_img = transforms.ToPILImage()(temp)
            temp_img.save(os.path.join(output_dir, f"output_step_{run[0]}.jpg"))

        run[0] += 1
        return total_loss

    optimizer.step(closure)

    with torch.no_grad():
        target.clamp_(-3, 3)

# Show Final Result

In [ ]:
plt.figure(figsize=(6,6))
imshow(target, "Stylized Output")
plt.show()

# Save Final Output

In [ ]:
final = target.clone().detach().cpu().squeeze(0)

mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

final = final * std + mean
final = torch.clamp(final, 0, 1)

final_img = transforms.ToPILImage()(final)
final_img.save(os.path.join(output_dir, "final_output.jpg"))

print("Final image saved.")

In [ ]:
!zip -r nst_outputs.zip "/content/drive/MyDrive/Colab Notebooks/GenAI/Lab 7/nst_outputs"